In [ ]:
import feedparser
import requests
import os
import time
from google import genai

In [ ]:
#默认获取最新一期播客，如果你想处理指定文件，可以修改逻辑

def get_latest_podcast_audio(rss_url): # old version, not used
    """解析 RSS 链接并获取最新一期播客的音频 URL 和标题"""
    print("正在解析 RSS 链接...")
    feed = feedparser.parse(rss_url)
    
    if not feed.entries:
        raise Exception("未找到任何播客内容，请检查 RSS 链接是否正确！")

    latest_episode = feed.entries[0]
    title = latest_episode.title

    # 遍历 links 寻找音频文件链接 (通常 rel 为 enclosure)
    audio_url = None
    for link in latest_episode.links:
        if link.rel == 'enclosure' and 'audio' in link.type:
            audio_url = link.href
            break

    if not audio_url:
        raise Exception("在最新一期节目中未找到音频下载链接！")

    print(f"找到最新一期播客: {title}")
    return title, audio_url

def get_podcast_by_index(rss_url, index=0):
    """
    根据索引获取播客：0 是最新，1 是往后推一期，以此类推
    """
    print(f"正在解析 RSS 链接并获取第 {index + 1} 期内容...")
    feed = feedparser.parse(rss_url)
    
    # 检查索引是否超出范围
    if not feed.entries or index >= len(feed.entries):
        raise Exception(f"索引超出范围！该播客源总共有 {len(feed.entries)} 期节目。")

    # 获取指定索引的文章
    target_episode = feed.entries[index]
    title = target_episode.title

    audio_url = None
    for link in target_episode.links:
        if link.rel == 'enclosure' and 'audio' in link.type:
            audio_url = link.href
            break

    if not audio_url:
        raise Exception(f"在第 {index + 1} 期节目中未找到音频下载链接！")

    print(f"找到节目: {title}")
    return title, audio_url

In [ ]:


def download_audio(audio_url, filename="temp_podcast.mp3"):
    """下载音频文件到本地"""
    print(f"正在下载音频 (可能需要几分钟，取决于文件大小): {audio_url}")
    response = requests.get(audio_url, stream=True)
    response.raise_for_status()
    
    with open(filename, 'wb') as f:
        # 分块下载以避免内存占用过大
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            
    print("✅ 音频下载完成！")
    return filename

def process_with_gemini(audio_path):
    """上传音频到 Gemini API 并获取逐字稿和翻译"""
    print("正在将音频上传至 Google 服务器 (Gemini File API)...")
    # 播客文件通常较大，必须使用 File API 上传
    client = genai.Client()

    # 正确的调用方式
    audio_file = client.files.upload(file = audio_path)
    #audio_file = genai.upload_file(path=audio_path)

    # 上传大文件后，Gemini 需要一点时间处理音频，需轮询等待状态变为 ACTIVE
    while audio_file.state.name == "PROCESSING":
        print("等待 Gemini 处理音频流...")
        time.sleep(10)
        # 刷新文件状态
        audio_file = genai.get_file(audio_file.name)

    if audio_file.state.name == "FAILED":
        raise Exception("Gemini 音频处理失败！")

    print("✅ 音频准备就绪，正在请求 AI 进行听写和翻译 (视音频长度可能需要 1-3 分钟)...")
    
    model = "gemini-2.5-flash"

    # 设计 Prompt 让模型一次性输出英文原文和中文翻译
    prompt = """
    Please listen to this English podcast audio carefully.
    
    Your task:
    1. Transcribe the entire audio into English text.
    2. Translate the transcribed English text into natural, fluent Simplified Chinese.

    Please output EXACTLY in the following format:

    [English Transcript]
    (Insert the full English transcript here)

    [Chinese Translation]
    (Insert the full Chinese translation here)
    """

    # 调用模型生成内容
    response = client.models.generate_content(model=model, contents = [audio_file, prompt])

    # 养成好习惯：处理完毕后从 Google 服务器删除临时文件以节省空间
    print("正在清理云端临时音频文件...")
    #genai.delete_file(audio_file.name)
    client.files.delete(name=audio_file.name)
    print(response.text)

    return response.text



In [ ]:
def main():
    rss_url = input("请输入英文播客的 RSS 链接: ").strip()

    try:
        # 获取用户想看第几期
        idx_input = input("你想获取从新往旧数的第几期？(直接回车代表第1期/最新): ").strip()
        
        # 转换输入为索引（用户输 1 对应索引 0）
        if idx_input == "":
            index = 0
        else:
            index = int(idx_input) - 1
            if index < 0:
                index = 0
        # 1. 获取并下载音频
        #title, audio_url = get_latest_podcast_audio(rss_url)
        title, audio_url = get_podcast_by_index(rss_url,index)
        audio_filename = "temp_podcast.mp3"
        download_audio(audio_url, audio_filename)

        # 2. 调用 Gemini API
        result_text = process_with_gemini(audio_filename)

        # 3. 将结果保存到文本文件
        # 清理标题中可能导致文件路径错误的特殊字符
        safe_title = "".join([c for c in title if c.isalpha() or c.isdigit() or c==' ']).rstrip()
        output_filename =f"{safe_title}_第{index+1}期_transcript.txt"
        
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(result_text)

        print(f"\n🎉 处理成功！文稿和翻译已保存至当前目录: {output_filename}")
    except ValueError:
        print("错误：请输入有效的数字序号！")
    except Exception as e:
        print(f"\n❌ 发生错误: {e}")
   
    # finally:
        
        # 4. 删除本地下载的临时音频文件
        # if os.path.exists("temp_podcast.mp3"):
        #     os.remove("temp_podcast.mp3")
        #     print("本地临时文件已清理。")

if __name__ == "__main__":
    main()